In [1]:
!pip install pandas
import pandas as pd


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [2]:
bt = pd.read_csv("bank_transactions.csv")
gl = pd.read_csv("general_ledger.csv")

In [4]:
gl

,datetime,amount,description,journal_entry_id
0,3/14/23 0:00,120000.00,INTERNET TRANSFER FROM DDA# 20020422222 ON 24 ...,c8bfe4be-e919-4b18-836e-4dc0de3e5436
1,9/25/23 0:00,64867.94,MOBILE DEPOSIT,eee04c6b-6dab-4eec-aff0-60df28d2d2e6
2,6/7/23 0:00,58000.00,NaN,118734a6-cea0-4514-a295-d1e59e78c6ca
3,4/12/23 0:00,58000.00,NaN,60ae8f8d-e075-40cd-b59e-298a892785f0
4,7/10/23 0:00,58000.00,NaN,ad69cf7c-0742-4293-ba60-26b3777e54c0
...,...,...,...,...
1821,6/28/23 0:00,-200000.00,ACH DEBIT FID BKG SVC LLC MONEYLINE MICO STRAT...,a50ebcde-4662-47cc-a960-9db656da7b58
1822,2/20/24 0:00,-200000.00,BILL 00/00/04 Payables Funding,fc16e07e-647c-457c-9553-a1e907b7b9eb
1823,10/20/23 0:00,-218826.36,BILL 20/00/02 Payables Funding,cbf1dd0e-dadb-47e9-ba46-7b5a67e0819c
1824,7/31/23 0:00,-223333.33,BILL 00/22/02 Payables Funding,c0150fea-a673-4d2c-9de6-bfc64ef44745


In [8]:
result = (
    gl.groupby('journal_entry_id', as_index=False)
      .agg({
          'description': 'first',
          'amount': 'sum'
      })
)

In [ ]:
bt

,datetime,amount,description
0,3/14/23 0:00,-1075000.00,ACH DEBIT FID BKG SVC LLC MONEYLINE MICO STRAT...
1,7/31/23 0:00,-223333.33,"ACH DEBIT BILL.COM PAYABLES MICO STRATEGIES, L..."
2,10/20/23 0:00,-218826.36,"ACH DEBIT BILL.COM PAYABLES MICO STRATEGIES, L..."
3,4/17/23 0:00,-200000.00,ACH DEBIT FID BKG SVC LLC MONEYLINE MICO STRAT...
4,6/28/23 0:00,-200000.00,ACH DEBIT FID BKG SVC LLC MONEYLINE MICO STRAT...
...,...,...,...
1952,3/8/23 0:00,58000.00,"INCOMING WIRESSQUARED HOLDINGS, LLC C/O JCT US..."
1953,4/12/23 0:00,58000.00,"INCOMING WIRESSQUARED HOLDINGS, LLC C/O JCT US..."
1954,9/25/23 0:00,64867.94,MOBILE DEPOSIT
1955,3/7/23 0:00,70000.00,MOBILE DEPOSIT


In [ ]:
result['amount']

,journal_entry_id,description,amount
0,002be6d6-4a51-44d2-874b-d64a62497204,TST* UPSIDE PIZZA BROOKLYN NY,-9.00
1,0046ca30-1984-47c7-a58e-721a0cf43d3b,POS PURCHASE #0020 APPLE.COM/BILL 2220200 CA,-10.88
2,009c43fb-4cc8-48c4-a775-69fbbd13f9d6,BILL 22/00/02 Payables Funding,-9905.08
3,00d3c25a-fe47-4cab-8194-b85994f6940c,MARYS FISH CAMP 0000NEW YORK NY,-162.00
4,00f89b4e-1cf3-4d42-8497-703030741279,SEAMLSS*BUSSTOPCAFE NEW YORK NY,-87.73
...,...,...,...
1769,ff49097f-a9df-4a5e-9011-0a1ada4df0d0,None,10000.00
1770,ff5f7834-031d-4b62-ad57-c3fb343ed16b,None,15000.00
1771,ffa1494a-d376-4af1-aadb-547cd9a05bf9,CAFE CLUNY NEW YORK NY,-110.54
1772,ffd22aa5-ca46-4e99-a967-f10451250775,None,20000.00


In [14]:
result.shape[0]

1774

In [ ]:
import numpy as np
from Levenshtein import distance

# ensure datetime columns are normalized where available
# (assumes `bt` and `gl` exist in the notebook namespace)
import pandas as pd

n_bt = bt.shape[0]
n_result = result.shape[0]

# normalize datetime columns (coerce errors to NaT)
if 'datetime' in bt.columns:
    bt['datetime'] = pd.to_datetime(bt['datetime'], errors='coerce')
else:
    bt['datetime'] = pd.NaT

if 'datetime' in gl.columns:
    gl['datetime'] = pd.to_datetime(gl['datetime'], errors='coerce')

# mapping: journal_entry_id -> representative entry datetime (first occurrence)
entry_dates = gl.groupby('journal_entry_id')['datetime'].first().to_dict()

source = 0
sink = n_bt + n_result + 1
N = sink + 1

# capacity matrix for Ford-Fulkerson
capacity = np.zeros((N, N), dtype=int)

# weight/cost matrix for edit distances
weight = np.full((N, N), np.inf)

# source -> bt nodes, capacity 1
for i in range(n_bt):
    bt_node = 1 + i
    capacity[source, bt_node] = 1
    weight[source, bt_node] = 0

# bt nodes -> result nodes: only create edge when the 15-day date-window rule is satisfied
for i, (_, trans) in enumerate(bt.iterrows()):
    bt_node = 1 + i

    for j, (_, ledger) in enumerate(result.iterrows()):
        result_node = 1 + n_bt + j

        desc1 = trans.get('description')
        desc2 = ledger.get('description')

        # Candidate date window: require both dates and |bank_date - entry_date| <= 15 days
        bank_date = trans.get('datetime')
        # prefer journal_entry_id from the aggregated `result` row if present
        journal_id = ledger.get('journal_entry_id') if 'journal_entry_id' in ledger.index else None
        entry_date = entry_dates.get(journal_id, pd.NaT)

        # coerce to Timestamps (pd.NaT if invalid)
        bank_date = pd.to_datetime(bank_date, errors='coerce')
        entry_date = pd.to_datetime(entry_date, errors='coerce')

        # skip if either date is missing
        if pd.isna(bank_date) or pd.isna(entry_date):
            continue

        # enforce 15-day window (inclusive)
        if abs((bank_date - entry_date).days) > 15:
            continue

        if desc1 is None or desc2 is None:
            continue

        # Amount closeness check: require |abs(bank_amount) - abs(entry_amount)| <= tol
        bank_amt = trans.get('amount')
        entry_amt = ledger.get('amount')

        # coerce to numeric; skip if missing or invalid
        bank_amt = pd.to_numeric(bank_amt, errors='coerce')
        entry_amt = pd.to_numeric(entry_amt, errors='coerce')
        if pd.isna(bank_amt) or pd.isna(entry_amt):
            continue

        tol = 0.01  # tolerance for rounding differences
        if abs(abs(bank_amt) - abs(entry_amt)) > tol:
            # amounts not sufficiently close -> infeasible edge
            continue

        capacity[bt_node, result_node] = 1
        weight[bt_node, result_node] = distance(str(desc1), str(desc2))

# result nodes -> sink, capacity 1
for j in range(n_result):
    result_node = 1 + n_bt + j
    capacity[result_node, sink] = 1
    weight[result_node, sink] = 0

In [33]:
def dfs(residual, s, t, visited, parent):
    visited[s] = True

    for v in range(len(residual)):
        if not visited[v] and residual[s][v] > 0:
            parent[v] = s
            if v == t:
                return True
            if dfs(residual, v, t, visited, parent):
                return True

    return False


def ford_fulkerson(capacity, source, sink):
    residual = capacity.copy()
    parent = [-1] * len(capacity)
    max_flow = 0

    while True:
        visited = [False] * len(capacity)
        parent = [-1] * len(capacity)

        if not dfs(residual, source, sink, visited, parent):
            break

        path_flow = float('inf')
        v = sink
        while v != source:
            u = parent[v]
            path_flow = min(path_flow, residual[u][v])
            v = u

        v = sink
        while v != source:
            u = parent[v]
            residual[u][v] -= path_flow
            residual[v][u] += path_flow
            v = u

        max_flow += path_flow

    return max_flow, residual

In [34]:
max_flow, residual = ford_fulkerson(capacity, source, sink)
print("Max flow:", max_flow)

Max flow: 1389


In [35]:
matches = []
for i in range(n_bt):
    for j in range(n_result):
        u = 1 + i
        v = 1 + n_bt + j
        if capacity[u, v] == 1 and residual[u, v] == 0:
            matches.append((i, j, weight[u, v]))

print(matches)

[(0, 1773, np.float64(45.0)), (1, 1771, np.float64(53.0)), (2, 1768, np.float64(53.0)), (3, 1767, np.float64(54.0)), (4, 1765, np.float64(0.0)), (5, 1763, np.float64(51.0)), (6, 1760, np.float64(57.0)), (7, 1756, np.float64(59.0)), (8, 1755, np.float64(57.0)), (9, 1754, np.float64(54.0)), (10, 1752, np.float64(53.0)), (11, 1751, np.float64(57.0)), (12, 1750, np.float64(80.0)), (13, 1749, np.float64(77.0)), (14, 1748, np.float64(57.0)), (15, 1747, np.float64(52.0)), (16, 1745, np.float64(54.0)), (17, 1744, np.float64(52.0)), (18, 1743, np.float64(53.0)), (19, 1740, np.float64(37.0)), (20, 1739, np.float64(51.0)), (21, 1738, np.float64(78.0)), (22, 1737, np.float64(54.0)), (23, 1734, np.float64(53.0)), (24, 1732, np.float64(59.0)), (25, 1731, np.float64(53.0)), (26, 1729, np.float64(49.0)), (27, 1728, np.float64(53.0)), (28, 1727, np.float64(51.0)), (29, 1726, np.float64(53.0)), (30, 1724, np.float64(47.0)), (31, 1723, np.float64(53.0)), (32, 1721, np.float64(60.0)), (33, 1720, np.float6

In [36]:
import json
import pandas as pd
from Levenshtein import ratio

# Example:
# matches = [(journal_idx, bank_idx), ...]
# or matches = [(journal_idx, bank_idx, something), ...]

output = []

for match in matches:
    journal_idx = match[0]
    bank_idx = match[1]

    gl_row = result.iloc[journal_idx]
    bank_row = bt.iloc[bank_idx]

    gl_desc = gl_row.get("description")
    bank_desc = bank_row.get("description")

    score = ratio(str(gl_desc or ""), str(bank_desc or ""))

    record = {
        "journal_entry_id": gl_row.get("journal_entry_id", str(journal_idx)),
        "gl_datetime": None if pd.isna(gl_row.get("datetime")) else str(gl_row.get("datetime")),
        "gl_amount": None if pd.isna(gl_row.get("amount")) else float(gl_row.get("amount")),
        "gl_description": None if pd.isna(gl_desc) else str(gl_desc),

        "bank_index": int(bank_idx),
        "bank_datetime": None if pd.isna(bank_row.get("datetime")) else str(bank_row.get("datetime")),
        "bank_amount": None if pd.isna(bank_row.get("amount")) else float(bank_row.get("amount")),
        "bank_description": None if pd.isna(bank_desc) else str(bank_desc),

        "score": float(score),
    }

    output.append(record)

with open("matches.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("Wrote matches.json")

Wrote matches.json
